# O1-09 — Observed vs Boundary-Like Pair Sensitivity

## Objective

The Gemfinder minimal-effects method evaluates a boundary version of the two
classroom histograms before Gamma-distribution inference.

The exact portal implementation and production epsilon value have not yet been
confirmed by RYA.

This notebook therefore performs a provisional sensitivity analysis by shrinking
the observed X/Y histograms toward their common midpoint at several controlled
levels.

The purpose is to determine whether Objective 1 routing is sensitive to this
transformation.

This is a robustness stress test only. The transformed pairs in this notebook
must not be described as the exact production boundary pairs until RYA confirms
the implementation.

In [2]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

BASE_DIR = Path(
    r"C:\AIML Y2 S2\Capstone 1\RYA"
)

DETECTOR_FILE = (
    BASE_DIR
    / "O1_07_compatibility_detector_v1.joblib"
)

METADATA_FILE = (
    BASE_DIR
    / "O1_07_compatibility_detector_v1_metadata.json"
)

TRAINING_FILE = (
    BASE_DIR
    / "O1_06_detector_training_high_conf_477.csv"
)

detector = joblib.load(
    DETECTOR_FILE
)

with open(
    METADATA_FILE,
    "r"
) as f:
    detector_metadata = json.load(f)

detector_data = pd.read_csv(
    TRAINING_FILE
)

feature_order = (
    detector_metadata[
        "feature_order"
    ]
)

threshold = float(
    detector_metadata[
        "threshold"
    ]
)

print(
    "Cases:",
    detector_data.shape
)

print(
    "Features:",
    len(feature_order)
)

print(
    "Threshold:",
    threshold
)

Cases: (477, 58)
Features: 54
Threshold: 0.1


In [4]:
import sys

if str(BASE_DIR) not in sys.path:
    sys.path.append(
        str(BASE_DIR)
    )

from feature_extractor import (
    extract_screening_features
)

In [6]:
def shrink_pair_to_midpoint(
    X,
    Y,
    s
):
    """
    Provisional boundary-like transformation.

    s = 1.0  -> original observed pair
    s < 1.0  -> X and Y moved toward their midpoint
    s = 0.0  -> identical midpoint distributions

    This follows the mathematical form described in the
    Gemfinder minimal-effects method, but does NOT claim
    to reproduce the production boundary until epsilon
    and the portal implementation are confirmed.
    """

    X = np.asarray(
        X,
        dtype=float
    )

    Y = np.asarray(
        Y,
        dtype=float
    )

    midpoint = (
        X + Y
    ) / 2.0

    X_s = (
        midpoint
        + s * (
            X - midpoint
        )
    )

    Y_s = (
        midpoint
        + s * (
            Y - midpoint
        )
    )

    return X_s, Y_s

In [8]:
X_example = np.array([
    0.50,
    0.20,
    0.20,
    0.10
])

Y_example = np.array([
    0.10,
    0.20,
    0.30,
    0.40
])

for s in [
    1.0,
    0.75,
    0.50,
    0.25,
    0.10
]:
    X_s, Y_s = (
        shrink_pair_to_midpoint(
            X_example,
            Y_example,
            s
        )
    )

    print(
        "\ns =",
        s
    )

    print(
        "X:",
        X_s
    )

    print(
        "Y:",
        Y_s
    )

    print(
        "distance:",
        np.linalg.norm(
            X_s - Y_s
        )
    )


s = 1.0
X: [0.5 0.2 0.2 0.1]
Y: [0.1 0.2 0.3 0.4]
distance: 0.5099019513592785

s = 0.75
X: [0.45   0.2    0.2125 0.1375]
Y: [0.15   0.2    0.2875 0.3625]
distance: 0.3824264635194589

s = 0.5
X: [0.4   0.2   0.225 0.175]
Y: [0.2   0.2   0.275 0.325]
distance: 0.25495097567963926

s = 0.25
X: [0.35   0.2    0.2375 0.2125]
Y: [0.25   0.2    0.2625 0.2875]
distance: 0.1274754878398196

s = 0.1
X: [0.32  0.2   0.245 0.235]
Y: [0.28  0.2   0.255 0.265]
distance: 0.050990195135927896


In [10]:
def detector_probability_for_pair(
    X,
    Y,
    n_x,
    n_y,
    P,
    detector,
    feature_order
):
    features = (
        extract_screening_features(
            X=X,
            Y=Y,
            n_x=n_x,
            n_y=n_y,
            P=P
        )
    )

    features[
        "euclid_dist_XY"
    ] = np.round(
        features[
            "euclid_dist_XY"
        ],
        4
    )

    series = (
        pd.Series(
            features,
            dtype=float
        )
        .reindex(
            feature_order
        )
    )

    input_df = pd.DataFrame(
        [
            series.values
        ],
        columns=feature_order
    )

    probability = float(
        detector.predict_proba(
            input_df
        )[0, 1]
    )

    route = (
        "fallback_required"
        if probability >= threshold
        else "pass_to_gamma_nn"
    )

    return probability, route

In [12]:
shrink_levels = [
    1.00,
    0.75,
    0.50,
    0.25,
    0.10
]

results = []

for _, row in detector_data.iterrows():

    X = np.array(
        [
            row[
                f"X_{i}"
            ]
            for i in range(
                1,
                5
            )
        ],
        dtype=float
    )

    Y = np.array(
        [
            row[
                f"Y_{i}"
            ]
            for i in range(
                1,
                5
            )
        ],
        dtype=float
    )

    P = np.array([
        [
            row["P_11"],
            row["P_12"],
            row["P_13"],
            row["P_14"]
        ],
        [
            row["P_21"],
            row["P_22"],
            row["P_23"],
            row["P_24"]
        ],
        [
            row["P_31"],
            row["P_32"],
            row["P_33"],
            row["P_34"]
        ],
        [
            row["P_41"],
            row["P_42"],
            row["P_43"],
            row["P_44"]
        ]
    ], dtype=float)

    for s in shrink_levels:

        X_s, Y_s = (
            shrink_pair_to_midpoint(
                X,
                Y,
                s
            )
        )

        probability, route = (
            detector_probability_for_pair(
                X=X_s,
                Y=Y_s,
                n_x=row["n_X"],
                n_y=row["n_Y"],
                P=P,
                detector=detector,
                feature_order=feature_order
            )
        )

        results.append({
            "exp_code":
                row["exp_code"],

            "compatibility_zone_v1":
                row[
                    "compatibility_zone_v1"
                ],

            "s":
                s,

            "probability":
                probability,

            "route":
                route
        })


boundary_sensitivity_df = pd.DataFrame(
    results
)

print(
    boundary_sensitivity_df.shape
)

display(
    boundary_sensitivity_df.head()
)

(2385, 5)


,exp_code,compatibility_zone_v1,s,probability,route
0,orthog_B_4_11483,compatible,1.00,0.000052,pass_to_gamma_nn
1,orthog_B_4_11483,compatible,0.75,0.000069,pass_to_gamma_nn
2,orthog_B_4_11483,compatible,0.50,0.000085,pass_to_gamma_nn
3,orthog_B_4_11483,compatible,0.25,0.000102,pass_to_gamma_nn
4,orthog_B_4_11483,compatible,0.10,0.000109,pass_to_gamma_nn


In [14]:
baseline = (
    boundary_sensitivity_df[
        boundary_sensitivity_df[
            "s"
        ] == 1.0
    ][
        [
            "exp_code",
            "probability",
            "route"
        ]
    ]
    .rename(
        columns={
            "probability":
                "baseline_probability",

            "route":
                "baseline_route"
        }
    )
)

comparison = (
    boundary_sensitivity_df
    .merge(
        baseline,
        on="exp_code",
        how="left"
    )
)

comparison[
    "probability_change"
] = (
    comparison[
        "probability"
    ]
    -
    comparison[
        "baseline_probability"
    ]
)

comparison[
    "absolute_probability_change"
] = (
    comparison[
        "probability_change"
    ].abs()
)

comparison[
    "route_changed"
] = (
    comparison[
        "route"
    ]
    !=
    comparison[
        "baseline_route"
    ]
)

In [16]:
sensitivity_summary = (
    comparison[
        comparison["s"] < 1.0
    ]
    .groupby(
        "s"
    )
    .agg(
        cases=(
            "exp_code",
            "count"
        ),

        mean_abs_probability_change=(
            "absolute_probability_change",
            "mean"
        ),

        median_abs_probability_change=(
            "absolute_probability_change",
            "median"
        ),

        p95_abs_probability_change=(
            "absolute_probability_change",
            lambda x:
                np.quantile(
                    x,
                    0.95
                )
        ),

        route_changes=(
            "route_changed",
            "sum"
        ),

        route_agreement=(
            "route_changed",
            lambda x:
                1.0
                -
                x.mean()
        )
    )
    .reset_index()
)

sensitivity_summary[
    "route_agreement_percent"
] = (
    sensitivity_summary[
        "route_agreement"
    ]
    * 100
)

display(
    sensitivity_summary
)

,s,cases,mean_abs_probability_change,median_abs_probability_change,p95_abs_probability_change,route_changes,route_agreement,route_agreement_percent
0,0.10,477,0.119082,0.006163,0.697598,44,0.907757,90.775681
1,0.25,477,0.110929,0.004881,0.672851,40,0.916143,91.614256
2,0.50,477,0.097774,0.003185,0.611272,34,0.928721,92.872117
3,0.75,477,0.084262,0.001742,0.586240,30,0.937107,93.710692


In [18]:
route_change_details = []

for s in [
    0.75,
    0.50,
    0.25,
    0.10
]:

    subset = comparison[
        comparison["s"]
        == s
    ]

    pass_to_fallback = (
        (
            subset[
                "baseline_route"
            ]
            ==
            "pass_to_gamma_nn"
        )
        &
        (
            subset[
                "route"
            ]
            ==
            "fallback_required"
        )
    ).sum()

    fallback_to_pass = (
        (
            subset[
                "baseline_route"
            ]
            ==
            "fallback_required"
        )
        &
        (
            subset[
                "route"
            ]
            ==
            "pass_to_gamma_nn"
        )
    ).sum()

    route_change_details.append({
        "s":
            s,

        "pass_to_fallback":
            int(
                pass_to_fallback
            ),

        "fallback_to_pass":
            int(
                fallback_to_pass
            )
    })


route_change_details_df = (
    pd.DataFrame(
        route_change_details
    )
)

display(
    route_change_details_df
)

,s,pass_to_fallback,fallback_to_pass
0,0.75,2,28
1,0.50,2,32
2,0.25,3,37
3,0.10,3,41


## Preliminary boundary-sensitivity finding

The provisional midpoint-shrink sensitivity analysis indicates that detector
routing is not completely invariant to changes in X/Y toward a boundary-like
pair.

Across 477 high-confidence cases, route agreement relative to the original
pair was:

- s = 0.75: 93.71%;
- s = 0.50: 92.87%;
- s = 0.25: 91.61%;
- s = 0.10: 90.78%.

More importantly, route changes were strongly asymmetric.

At s = 0.10, 44 cases changed route:

- 3 changed from pass to fallback;
- 41 changed from fallback to pass.

This suggests that shrinking the histograms toward one another can make some
cases appear safer to the current compatibility classifier.

This result is provisional because the exact production boundary
transformation and epsilon value have not yet been confirmed.

Before changing or retraining the classifier, it is necessary to establish
whether the X/Y pairs in the supplied RYA instance dataset already represent
the boundary inputs used by the Gamma neural network.

In [21]:
baseline_route_counts = (
    baseline[
        "baseline_route"
    ]
    .value_counts()
)

print(
    "Baseline route counts:"
)

print(
    baseline_route_counts
)

Baseline route counts:
baseline_route
pass_to_gamma_nn     266
fallback_required    211
Name: count, dtype: int64


In [23]:
fallback_sensitivity = []

baseline_fallback_count = int(
    (
        baseline[
            "baseline_route"
        ]
        == "fallback_required"
    ).sum()
)

for s in [
    0.75,
    0.50,
    0.25,
    0.10
]:

    subset = comparison[
        comparison["s"] == s
    ]

    fallback_to_pass = int(
        (
            (
                subset["baseline_route"]
                == "fallback_required"
            )
            &
            (
                subset["route"]
                == "pass_to_gamma_nn"
            )
        ).sum()
    )

    fallback_sensitivity.append({
        "s": s,
        "baseline_fallback_cases":
            baseline_fallback_count,
        "fallback_to_pass":
            fallback_to_pass,
        "fallback_retained":
            baseline_fallback_count
            - fallback_to_pass,
        "fallback_retention_percent":
            100
            * (
                baseline_fallback_count
                - fallback_to_pass
            )
            / baseline_fallback_count,
        "fallback_lost_percent":
            100
            * fallback_to_pass
            / baseline_fallback_count
    })


fallback_sensitivity_df = pd.DataFrame(
    fallback_sensitivity
)

display(
    fallback_sensitivity_df
)

,s,baseline_fallback_cases,fallback_to_pass,fallback_retained,fallback_retention_percent,fallback_lost_percent
0,0.75,211,28,183,86.729858,13.270142
1,0.50,211,32,179,84.834123,15.165877
2,0.25,211,37,174,82.464455,17.535545
3,0.10,211,41,170,80.568720,19.431280


In [25]:
changed_case_label_summary = (
    comparison[
        comparison["route_changed"]
        &
        (comparison["s"] < 1.0)
    ]
    .groupby(
        [
            "s",
            "compatibility_zone_v1",
            "baseline_route",
            "route"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

display(
    changed_case_label_summary
)

,s,compatibility_zone_v1,baseline_route,route,count
0,0.10,compatible,fallback_required,pass_to_gamma_nn,27
1,0.10,compatible,pass_to_gamma_nn,fallback_required,3
2,0.10,incompatible,fallback_required,pass_to_gamma_nn,14
3,0.25,compatible,fallback_required,pass_to_gamma_nn,26
4,0.25,compatible,pass_to_gamma_nn,fallback_required,3
5,0.25,incompatible,fallback_required,pass_to_gamma_nn,11
6,0.50,compatible,fallback_required,pass_to_gamma_nn,21
7,0.50,compatible,pass_to_gamma_nn,fallback_required,2
8,0.50,incompatible,fallback_required,pass_to_gamma_nn,11
9,0.75,compatible,fallback_required,pass_to_gamma_nn,18


In [27]:
# Baseline performance by true compatibility label

baseline_with_labels = (
    comparison[
        comparison["s"] == 1.0
    ][
        [
            "exp_code",
            "compatibility_zone_v1",
            "baseline_probability",
            "baseline_route"
        ]
    ]
    .drop_duplicates(
        subset="exp_code"
    )
)

baseline_label_route_table = pd.crosstab(
    baseline_with_labels[
        "compatibility_zone_v1"
    ],
    baseline_with_labels[
        "baseline_route"
    ]
)

display(
    baseline_label_route_table
)

baseline_route,fallback_required,pass_to_gamma_nn
compatibility_zone_v1,,
compatible,67,266
incompatible,144,0


In [29]:
incompatible_sensitivity = []

for s in [
    1.00,
    0.75,
    0.50,
    0.25,
    0.10
]:

    subset = comparison[
        (
            comparison["s"] == s
        )
        &
        (
            comparison[
                "compatibility_zone_v1"
            ]
            == "incompatible"
        )
    ]

    total_incompatible = len(
        subset
    )

    routed_fallback = int(
        (
            subset["route"]
            == "fallback_required"
        ).sum()
    )

    routed_nn = int(
        (
            subset["route"]
            == "pass_to_gamma_nn"
        ).sum()
    )

    incompatible_sensitivity.append({
        "s":
            s,

        "total_incompatible":
            total_incompatible,

        "fallback_required":
            routed_fallback,

        "pass_to_gamma_nn":
            routed_nn,

        "incompatible_recall_percent":
            100
            * routed_fallback
            / total_incompatible
    })


incompatible_sensitivity_df = pd.DataFrame(
    incompatible_sensitivity
)

display(
    incompatible_sensitivity_df
)

,s,total_incompatible,fallback_required,pass_to_gamma_nn,incompatible_recall_percent
0,1.00,144,144,0,100.000000
1,0.75,144,134,10,93.055556
2,0.50,144,133,11,92.361111
3,0.25,144,133,11,92.361111
4,0.10,144,130,14,90.277778


### Boundary-like transformation sensitivity

The current compatibility detector is sensitive to provisional midpoint-shrink
transformations of X/Y.

Using the 144 high-confidence incompatible cases:

- original pairs: 144/144 detected (100.0% recall);
- s = 0.75: 134/144 detected (93.1%);
- s = 0.50: 133/144 detected (92.4%);
- s = 0.25: 133/144 detected (92.4%);
- s = 0.10: 130/144 detected (90.3%).

This indicates that shrinking the response histograms toward one another can
reduce the detector's ability to identify some Gamma-incompatible cases.

The result is provisional because the exact production boundary construction
and epsilon value have not yet been confirmed.

No retraining or threshold change is made at this stage. The next step is to
confirm whether the supplied RYA instance X/Y pairs already correspond to the
boundary inputs received by the Gamma neural network.

## 4. Characterise incompatible cases lost after boundary-like shrinking

The provisional boundary sensitivity experiment showed that some
high-confidence Gamma-incompatible cases move below the O1 routing threshold
after X/Y are shrunk toward their midpoint.

This section identifies those cases and examines whether the lost cases share
specific characteristics such as extreme group-size imbalance, sparse
histograms or probabilities close to the routing threshold.

In [33]:
lost_incompatible_s010 = comparison[
    (comparison["s"] == 0.10)
    &
    (
        comparison["compatibility_zone_v1"]
        == "incompatible"
    )
    &
    (
        comparison["baseline_route"]
        == "fallback_required"
    )
    &
    (
        comparison["route"]
        == "pass_to_gamma_nn"
    )
].copy()

print(
    "Lost incompatible cases at s=0.10:",
    len(lost_incompatible_s010)
)

Lost incompatible cases at s=0.10: 14


In [35]:
lost_incompatible_detail = (
    lost_incompatible_s010
    .merge(
        detector_data,
        on=[
            "exp_code",
            "compatibility_zone_v1"
        ],
        how="left",
        suffixes=(
            "_sensitivity",
            ""
        )
    )
)

lost_incompatible_detail[
    "group_size_ratio_check"
] = (
    lost_incompatible_detail[
        [
            "n_X",
            "n_Y"
        ]
    ]
    .max(axis=1)
    /
    lost_incompatible_detail[
        [
            "n_X",
            "n_Y"
        ]
    ]
    .min(axis=1)
)

In [37]:
cols_to_show = [
    "exp_code",
    "P_matrix_question",
    "n_X",
    "n_Y",
    "group_size_ratio_check",
    "sparse_count_X",
    "sparse_count_Y",
    "degenerate_X",
    "degenerate_Y",
    "euclid_dist_XY",
    "baseline_probability",
    "probability",
    "absolute_probability_change"
]

display(
    lost_incompatible_detail[
        cols_to_show
    ]
    .sort_values(
        "group_size_ratio_check",
        ascending=False
    )
)

,exp_code,P_matrix_question,n_X,n_Y,group_size_ratio_check,sparse_count_X,sparse_count_Y,degenerate_X,degenerate_Y,euclid_dist_XY,baseline_probability,probability,absolute_probability_change
6,orthog_B_1_4161,for2,6,623,103.833333,3,0,1,0,0.9618,0.969448,0.073333,0.896115
3,orthog_B_1_23695,cop3,5,442,88.400000,3,0,1,0,0.6363,0.994388,0.034514,0.959874
4,orthog_B_4_25640,ry11,9,635,70.555556,3,0,1,0,1.0022,0.906927,0.030963,0.875964
5,orthog_B_1_15208,sun12,5,76,15.200000,3,0,1,0,1.0221,0.875281,0.011882,0.863400
7,orthog_B_1_14027,ry17,61,5,12.200000,0,3,0,1,0.7848,0.693605,0.013381,0.680224
11,orthog_B_1_18481,ry28,5,43,8.600000,3,0,1,0,0.9401,0.991295,0.033936,0.957360
10,orthog_B_1_27662,tru1,5,41,8.200000,2,0,0,0,0.7069,0.709052,0.060992,0.648060
12,orthog_B_1_9115,ry18,5,35,7.000000,3,0,1,0,0.7814,0.884170,0.004359,0.879811
8,orthog_B_1_5034,cop4,6,40,6.666667,3,0,1,0,0.6315,0.988514,0.016614,0.971900
1,orthog_B_1_15813,ry15,9,59,6.555556,3,0,1,0,0.9305,0.568573,0.001341,0.567232


In [39]:
print(
    "Group-size ratio summary:"
)

display(
    lost_incompatible_detail[
        "group_size_ratio_check"
    ].describe()
)

print(
    "\nRatio >= 10:",
    (
        lost_incompatible_detail[
            "group_size_ratio_check"
        ] >= 10
    ).sum()
)

print(
    "Ratio >= 50:",
    (
        lost_incompatible_detail[
            "group_size_ratio_check"
        ] >= 50
    ).sum()
)

print(
    "Ratio >= 100:",
    (
        lost_incompatible_detail[
            "group_size_ratio_check"
        ] >= 100
    ).sum()
)

Group-size ratio summary:


count     14.000000
mean      24.884127
std       34.723972
min        5.000000
25%        5.988889
50%        7.600000
75%       14.450000
max      103.833333
Name: group_size_ratio_check, dtype: float64


Ratio >= 10: 5
Ratio >= 50: 3
Ratio >= 100: 1


In [41]:
display(
    lost_incompatible_detail[
        [
            "exp_code",
            "baseline_probability",
            "probability",
            "absolute_probability_change"
        ]
    ]
    .sort_values(
        "baseline_probability"
    )
)

,exp_code,baseline_probability,probability,absolute_probability_change
1,orthog_B_1_15813,0.568573,0.001341,0.567232
13,orthog_B_4_17747,0.656480,0.000289,0.656191
7,orthog_B_1_14027,0.693605,0.013381,0.680224
10,orthog_B_1_27662,0.709052,0.060992,0.648060
9,orthog_B_4_11742,0.862730,0.036073,0.826656
5,orthog_B_1_15208,0.875281,0.011882,0.863400
12,orthog_B_1_9115,0.884170,0.004359,0.879811
2,orthog_B_1_2802,0.889919,0.067303,0.822617
4,orthog_B_4_25640,0.906927,0.030963,0.875964
0,orthog_B_1_11029,0.962024,0.076605,0.885419


In [43]:
lost_incompatible_all = comparison[
    (comparison["s"] < 1.0)
    &
    (
        comparison["compatibility_zone_v1"]
        == "incompatible"
    )
    &
    (
        comparison["baseline_route"]
        == "fallback_required"
    )
    &
    (
        comparison["route"]
        == "pass_to_gamma_nn"
    )
].copy()

lost_frequency = (
    lost_incompatible_all
    .groupby(
        "exp_code"
    )
    .agg(
        shrink_levels_lost=(
            "s",
            "count"
        ),
        min_s=(
            "s",
            "min"
        ),
        max_s=(
            "s",
            "max"
        )
    )
    .sort_values(
        "shrink_levels_lost",
        ascending=False
    )
)

print(
    "Unique incompatible cases lost at any shrink level:",
    len(lost_frequency)
)

display(
    lost_frequency
)

Unique incompatible cases lost at any shrink level: 15


,shrink_levels_lost,min_s,max_s
exp_code,,,
orthog_B_1_14027,4,0.1,0.75
orthog_B_1_15208,4,0.1,0.75
orthog_B_1_15813,4,0.1,0.75
orthog_B_1_27662,4,0.1,0.75
orthog_B_1_4161,4,0.1,0.75
orthog_B_1_5034,4,0.1,0.75
orthog_B_1_9115,4,0.1,0.75
orthog_B_4_17747,4,0.1,0.75
orthog_B_4_25640,4,0.1,0.75


### Characterisation of incompatible cases lost after shrinking

The boundary-like sensitivity effect is concentrated in a small and
structurally distinctive subgroup.

Across all tested shrink levels, only 15 unique incompatible cases ever
changed from fallback_required to pass_to_gamma_nn.

At s = 0.10, 14 incompatible cases were missed. These were not borderline
detector cases: their original incompatibility probabilities ranged from
approximately 0.57 to 0.99.

The dominant common feature was sparse or degenerate response structure.
Thirteen of the fourteen cases had a degenerate histogram in one classroom,
and all fourteen contained substantial sparsity in at least one classroom.

Group-size imbalance was also present, but varied substantially:
the group-size ratio ranged from 5.0 to approximately 103.8.

This indicates that the provisional boundary transformation can remove
response-shape signals that the current detector uses to recognise
Gamma-incompatible sparse/degenerate cases.

The effect is concentrated rather than widespread: only 15 unique
incompatible cases were affected across all four tested shrink levels,
with nine cases affected at every level.

No classifier change is made yet because the production boundary
transformation has not been confirmed.

In [46]:
degenerate_rule_eval = detector_data.copy()

degenerate_rule_eval[
    "degenerate_any"
] = (
    (
        degenerate_rule_eval[
            "degenerate_X"
        ] == 1
    )
    |
    (
        degenerate_rule_eval[
            "degenerate_Y"
        ] == 1
    )
)

degenerate_summary = (
    pd.crosstab(
        degenerate_rule_eval[
            "compatibility_zone_v1"
        ],
        degenerate_rule_eval[
            "degenerate_any"
        ],
        margins=True
    )
)

display(
    degenerate_summary
)

degenerate_any,False,True,All
compatibility_zone_v1,,,
compatible,325,8,333
incompatible,109,35,144
All,434,43,477


In [48]:
compatible_df = (
    degenerate_rule_eval[
        degenerate_rule_eval[
            "compatibility_zone_v1"
        ] == "compatible"
    ]
)

incompatible_df = (
    degenerate_rule_eval[
        degenerate_rule_eval[
            "compatibility_zone_v1"
        ] == "incompatible"
    ]
)

compatible_flagged = int(
    compatible_df[
        "degenerate_any"
    ].sum()
)

incompatible_flagged = int(
    incompatible_df[
        "degenerate_any"
    ].sum()
)

print(
    "Compatible cases flagged by degenerate rule:",
    compatible_flagged,
    "/",
    len(compatible_df)
)

print(
    "Incompatible cases captured by degenerate rule:",
    incompatible_flagged,
    "/",
    len(incompatible_df)
)

Compatible cases flagged by degenerate rule: 8 / 333
Incompatible cases captured by degenerate rule: 35 / 144


In [50]:
# Original-pair degeneracy flag by case
original_flags = (
    detector_data[
        [
            "exp_code",
            "compatibility_zone_v1",
            "degenerate_X",
            "degenerate_Y"
        ]
    ]
    .copy()
)

original_flags["original_degenerate_any"] = (
    (original_flags["degenerate_X"] == 1)
    |
    (original_flags["degenerate_Y"] == 1)
)

combined = comparison.merge(
    original_flags[
        [
            "exp_code",
            "original_degenerate_any"
        ]
    ],
    on="exp_code",
    how="left"
)

# Combined routing:
# fallback if either
#   1. original pair is degenerate, OR
#   2. transformed-pair detector requests fallback
combined["combined_route"] = np.where(
    combined["original_degenerate_any"]
    |
    (
        combined["route"]
        == "fallback_required"
    ),
    "fallback_required",
    "pass_to_gamma_nn"
)

In [52]:
combined_incompatible_summary = []

for s in [
    1.00,
    0.75,
    0.50,
    0.25,
    0.10
]:

    subset = combined[
        (
            combined["s"] == s
        )
        &
        (
            combined[
                "compatibility_zone_v1"
            ]
            == "incompatible"
        )
    ]

    total = len(subset)

    caught = int(
        (
            subset["combined_route"]
            == "fallback_required"
        ).sum()
    )

    missed = total - caught

    combined_incompatible_summary.append({
        "s": s,
        "total_incompatible": total,
        "fallback_required": caught,
        "pass_to_gamma_nn": missed,
        "combined_recall_percent":
            100 * caught / total
    })

combined_incompatible_summary_df = pd.DataFrame(
    combined_incompatible_summary
)

display(
    combined_incompatible_summary_df
)

,s,total_incompatible,fallback_required,pass_to_gamma_nn,combined_recall_percent
0,1.00,144,144,0,100.000000
1,0.75,144,142,2,98.611111
2,0.50,144,142,2,98.611111
3,0.25,144,143,1,99.305556
4,0.10,144,143,1,99.305556


In [54]:
combined_compatible_summary = []

for s in [
    1.00,
    0.75,
    0.50,
    0.25,
    0.10
]:

    subset = combined[
        (
            combined["s"] == s
        )
        &
        (
            combined[
                "compatibility_zone_v1"
            ]
            == "compatible"
        )
    ]

    total = len(subset)

    fallback = int(
        (
            subset["combined_route"]
            == "fallback_required"
        ).sum()
    )

    passed = total - fallback

    combined_compatible_summary.append({
        "s": s,
        "total_compatible": total,
        "fallback_required": fallback,
        "pass_to_gamma_nn": passed,
        "compatible_fallback_percent":
            100 * fallback / total
    })

combined_compatible_summary_df = pd.DataFrame(
    combined_compatible_summary
)

display(
    combined_compatible_summary_df
)

,s,total_compatible,fallback_required,pass_to_gamma_nn,compatible_fallback_percent
0,1.00,333,68,265,20.420420
1,0.75,333,57,276,17.117117
2,0.50,333,53,280,15.915916
3,0.25,333,51,282,15.315315
4,0.10,333,50,283,15.015015


### Candidate prototype safeguard

A simple original-pair degeneracy rule appears promising.

Among the 477 high-confidence labelled cases:

- 35/144 incompatible cases were degenerate in at least one classroom;
- only 8/333 compatible cases were degenerate.

Therefore, using original-pair degeneracy as an additional fallback trigger
has relatively low apparent cost.

This safeguard must be evaluated using the observed pair before any boundary
shrink, because the boundary transformation may remove the exact degeneracy
signal.

The final decision should therefore depend on whether the production pipeline
can expose both the observed pair and the boundary pair to Objective 1.

## O1-09 Final provisional conclusion

The compatibility classifier is sensitive to boundary-like shrinking of the
response histograms. Incompatible-case recall fell from 100% on the original
pairs to between 90.3% and 93.1% under the tested shrink transformations.

Further investigation showed that the affected cases were concentrated in
sparse and degenerate classroom distributions.

A simple safeguard was therefore evaluated:

1. inspect the original observed X/Y pair;
2. if either histogram is degenerate, route directly to fallback;
3. otherwise, run the compatibility classifier on the X/Y pair supplied to
   the Gamma neural network.

This combined rule substantially improved robustness.

Across 144 high-confidence incompatible cases:

- s = 1.00: 100.0% recall;
- s = 0.75: 98.6% recall;
- s = 0.50: 98.6% recall;
- s = 0.25: 99.3% recall;
- s = 0.10: 99.3% recall.

At s = 0.10, only 1 of 144 incompatible cases passed to the Gamma pathway,
compared with 14 cases using the transformed-pair classifier alone.

The compatible-case fallback rate under the tested shrink scenarios was
approximately 15–17%.

This provides a practical prototype safeguard while the exact production
boundary transformation remains unconfirmed.

The safeguard should remain provisional until RYA confirms:
- where the boundary transformation occurs;
- the production epsilon value;
- whether the original observed X/Y pair remains available to the screening
  layer before transformation.

In [58]:
OUTPUT_DIR = BASE_DIR

sensitivity_summary.to_csv(
    OUTPUT_DIR / "O1_09_classifier_only_sensitivity.csv",
    index=False
)

combined_incompatible_summary_df.to_csv(
    OUTPUT_DIR / "O1_09_combined_rule_incompatible_summary.csv",
    index=False
)

combined_compatible_summary_df.to_csv(
    OUTPUT_DIR / "O1_09_combined_rule_compatible_summary.csv",
    index=False
)

print("Saved:")
print("O1_09_classifier_only_sensitivity.csv")
print("O1_09_combined_rule_incompatible_summary.csv")
print("O1_09_combined_rule_compatible_summary.csv")

Saved:
O1_09_classifier_only_sensitivity.csv
O1_09_combined_rule_incompatible_summary.csv
O1_09_combined_rule_compatible_summary.csv
